# Cross validation

> In this example we'll see how to implement time series cross-validation 

::: {.callout-warning collapse="true"}

## Prerequesites

This tutorial assumes basic familiarity with StatsForecast. For a minimal example visit the [Quick Start](https://nixtla.github.io/statsforecast/examples/getting_started_short.html) 
:::

## Introduction 
**Cross-validation** is a widely used technique in Data Science for training and testing a model. Instead of using a single train-test split, we use several to better capture the performance of our model. Cross-validation usually involves shuffling the data, so a special version is necessary for handling time series. In this notebook, we'll explain how this works and how to implement it on [StatsForecast](https://nixtla.github.io/statsforecast/). 

**Outline:**

1. Install libraries 
2. Load and explore the data 
3. Prepare dataset for StatsForecast modeling
4. Understand time series cross-validation 
4. Generate rolling window AutoARIMA predictions 
5. Compute accuracy 
6. Plot predictions 

::: {.callout-tip}
You can use Colab to run this Notebook interactively <a href="https://colab.research.google.com/github/Nixtla/statsforecast/blob/main/nbs/examples/CrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
::: 

## Install libraries 

We assume that you have StatsForecast already installed. If not, check this guide for instructions on [how to install StatsForecast](https://nixtla.github.io/statsforecast/examples/installation.html) 

Install the necessary packages using `pip install statsforecast`

In [ ]:
pip install statsforecast 

In [ ]:
import random
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from neuralforecast.losses.numpy import mqloss
from sklearn import metrics
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

plt.rcParams["figure.figsize"] = (9,6)

### Auxiliar plot functions

In [ ]:
# Auxiliar plot functions
def plot_cv_indices(cv_indices, total_obs, ax, n_splits, lw=10):
    """Create a sample plot for indices of a cross-validation object."""

    colors = {0:'tab:blue', 1:'tab:orange', 2:'tab:gray'}
    # Generate the training/testing visualizations for each CV split
    for ii, (tr, tt) in enumerate(cv_indices):
        # Fill in indices with the training/test groups
        indices = np.array([2] * total_obs)
        indices[tt] = 1
        indices[tr] = 0

        # Visualize the results
        ax.scatter(
            range(len(indices)),
            [ii + 0.5] * len(indices),
            c=pd.Series(indices).map(colors),
            marker="_",
            lw=lw,
            # cmap=cmap_cv,
            vmin=-0.2,
            vmax=1.2,
        )

    # Formatting
    yticklabels = list(range(n_splits))
    ax.set(
        yticks=np.arange(n_splits) + 0.5,
        yticklabels=yticklabels,
        xlabel="Index",
        ylabel="CV iteration",
        ylim=[n_splits, -0.2],
        xlim=[0, total_obs],
    )
    ax.set_title("Cross-validation splits", fontsize=15)
    return ax

## Load and explore the data 

The dataset we'll be using in this notebook is an Australian Tourism dataset containing quarterly data on the number of trips performed between 1998 and 2017. The trips are categorized by Region, State, and Purpose. Our goal is to forecast the number of trips at State level. 

In [ ]:
data_url = "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/main/datasets/tourism.csv"
tourism_df = pd.read_csv(data_url, sep=",")
tourism_df.head()

We can see that there are 8 different States

In [ ]:
tourism_df['State'].unique()

## Prepare dataset for `StatsForecast` modelling

The input to StatsForecast is always a data frame in [long format](https://www.theanalysisfactor.com/wide-and-long-data/) with three columns: `unique_id`, `ds`, and `y`. 

* The `unique_id` (string, int, or category) column is a unique identifier for the series. 
* The `ds` (int or datestamp) column is either an integer indexing time or a datestamp in format YYYY-MM-DD or YYYY-MM-DD HH:MM:SS. 
* The `y` (numeric) column is the measurement we want to forecast, in this case, the number of trips. 

Since our goal is to forecast at State level, we first need to aggregate the data at this level. We can ignore the variables of Region and Purpose since they are not relevent to this example. 

In [ ]:
# Aggregate
target_var = "Trips"
aggregation_vars = ["Quarter", "State"] 
tourism_df_agg = tourism_df[aggregation_vars + [target_var]].groupby(aggregation_vars, as_index=False).sum()

# Create a dict maping string values in Quarter to datetime
quarters = tourism_df["Quarter"].sort_values().unique()
ds = pd.date_range(start='1998-01-01', periods=len(quarters), freq='Q')
ds_quarter = dict(zip(quarters, ds))

# Prepare columns
tourism_df_agg.rename(columns={target_var: 'y'}, inplace=True)
tourism_df_agg["unique_id"] = tourism_df_agg.groupby("State").ngroup()
tourism_df_agg["ds"] = [ds_quarter[q] for q in tourism_df_agg["Quarter"]]

State_id = dict(zip(tourism_df_agg["unique_id"].unique(), tourism_df_agg["State"].unique()))
tourism_df_agg.drop(["Quarter", "State"], axis=1, inplace=True)

In [ ]:
tourism_df_agg.head()

We can now plot the number of quarterly trips to each State

In [ ]:
StatsForecast.plot(df=tourism_df_agg)

To see the State associated with a particular id, simply use the following command. 

In [ ]:
State_id

## Understand time series cross-validation

Generally speaking, cross-validation consists of the following steps: 

1. Split the available information into $K$ partitions or folds.  
2. Select $K-1$ partitions to train the model and use the remaining partition (test set) for evaluation. 
3. Compute the accuracy of the model as the average of the $K$ validation scores obtained in each test set. 

From this description, it should be clear that when dealing with time series data, each partition can only consists of observations that occurred _prior_ to the observartions included in the test set. The following diagram shows how these partitions are defined, with the blue observations representing the training sets, and the orange observations the test sets. 

In [ ]:
total_obs = 100
nfolds = 10
cv_indices =[]

for i in range(1, nfolds+1)[::-1]:
    cutoff = 100 - i*5
    cv_indices.append((np.array(range(cutoff)), np.array(range(cutoff, cutoff+5))))

fig, ax = plt.subplots()
plot_cv_indices(cv_indices, total_obs, ax, nfolds)

This type of cross-validation is also known as _evaluation on a rolling origin or a rolling window_, since the point where the forecast starts rolls forward in time.

## Generate rolling window AutoARIMA predictions

To generate the forecasts in each cross-validation fold, we'll use StatsForecast's [AutoARIMA](https://nixtla.github.io/statsforecast/models.html#autoarima), [a reliable and computationally efficient method](https://nixtla.github.io/statsforecast/examples/getting_started_with_auto_arima_and_ets.html).

We'll need to: 

1. Select the modelling parameters. 
2. Create the forecast object. 
3. Perform the cross-validation.

In [ ]:
# Modelling parameters
season_length = 4
horizon = 1
freq = "Q"
n_windows_cv = 20

fcst = StatsForecast(
    df=tourism_df_agg, 
    models=[AutoARIMA(season_length=season_length)], 
    freq=freq, 
    n_jobs=-1
)

forecasts_cv = fcst.cross_validation(h=horizon, n_windows=n_windows_cv, level=(90,)) 
forecasts_cv.head()

We can plot the cross-validation splits as follows. 

In [ ]:
time_cv_indices = [
    (np.where(ds <= cutoff), np.where(ds == cutoff) + np.array(1)) \
    for cutoff in forecasts_cv[forecasts_cv.index == 0]["cutoff"]
]

fig, ax = plt.subplots()
plot_cv_indices(time_cv_indices, tourism_df_agg[tourism_df_agg.unique_id == 0].shape[0], ax, n_windows_cv)

## Compute accuracy 

To evaluate our model, we'll use the average of the Mean Absolute Error (MAE) over all the cross-validation folds, defined as: 

$$ \mathrm{MAE}(y, \hat{y}) = \frac{1}{N*H} \sum_{i,\tau} |y_{i,\tau}-\hat{y}_{i,\tau}| $$

In [ ]:
def compute_MAE(df, model_name):
    return metrics.mean_absolute_error(df["y"], df[model_name])

cutoff_values = forecasts_cv["cutoff"].unique()
cv_MAE = []
for ct in cutoff_values:
    cv_MAE.append(compute_MAE(forecasts_cv[forecasts_cv["cutoff"] == ct], "AutoARIMA"))
print(f"Average Mean Absolute Error across all Cross-Validation folds: {str(np.round(np.mean(cv_MAE), decimals=2))}")

## Plot predictions 

To conclude, we can plot the forecasts for each State. 

In [ ]:
forecasts_cv.rename(
    columns={"y":"True"}, 
    inplace=True
)

StatsForecast.plot(df=tourism_df_agg, forecasts_df=forecasts_cv.drop(columns=['cutoff']))

## References 

[Rob J. Hyndman and George Athanasopoulos (2018). "Forecasting principles and practice, Time series cross-validation"](https://otexts.com/fpp3/tscv.html).